In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import os
os.listdir()

['.ipynb_checkpoints', 'ArachneX_analysis.ipynb']

In [3]:
df = pd.read_csv(r"C://Users//123456//Desktop//Arachne//data//train.cv") 
df = df[["Dates", "Category", "X", "Y"]]

df = df.rename(columns={
    "Dates": "date",
    "Category": "crime_type",
    "X": "longitude",
    "Y": "latitude"
})

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df.head()

ImportError: Missing optional dependency 'fsspec'.  Use pip or conda to install fsspec.

In [ ]:
df["week"] = df["date"].dt.to_period("W")

In [ ]:
df["lat_bin"] = df["latitude"].round(3)
df["lon_bin"] = df["longitude"].round(3)

In [ ]:
weekly_counts = (
    df.groupby(["lat_bin", "lon_bin", "week"])
    .size()
    .reset_index(name="crime_count")
)

weekly_counts.head()

In [ ]:
weekly_counts["previous_week"] = (
    weekly_counts.groupby(["lat_bin", "lon_bin"])["crime_count"]
    .shift(1)
)

weekly_counts["previous_week"] = weekly_counts["previous_week"].fillna(0)

In [ ]:
weekly_counts["percent_increase"] = (
    (weekly_counts["crime_count"] - weekly_counts["previous_week"])
    / (weekly_counts["previous_week"] + 1)
) * 100

In [ ]:
weekly_counts["spike"] = weekly_counts["percent_increase"] > 100

In [ ]:
weekly_counts["risk_score"] = (
    weekly_counts["crime_count"] *
    (1 + weekly_counts["percent_increase"] / 100)
)

In [ ]:
spikes = weekly_counts[weekly_counts["spike"] == True]

print("Total Spikes:", len(spikes))

spikes.head()

In [ ]:
spikes = spikes.sort_values("risk_score", ascending=False).head(1000)

In [ ]:
heat_data = spikes[["lat_bin", "lon_bin", "risk_score"]].values.tolist()

sf_map = folium.Map(location=[37.77, -122.42], zoom_start=12)

HeatMap(
    heat_data,
    radius=15,
    blur=20,
    max_zoom=13
).add_to(sf_map)

sf_map